# 04 — RGB Composites from NISAR GCOV

Generate false-color RGB composites from dual-pol (HH/HV) GCOV backscatter data using
12 built-in methods. Demonstrates `make_rgb()` dispatcher, `list_rgb_methods()`,
contrast enhancement, and GeoTIFF export.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bullocke/nice-sar/blob/main/notebooks/04_rgb_composites.ipynb)

## 1. Install and Import

In [ ]:
# Pin fsspec/s3fs to avoid Colab conflicts, then force-reinstall only nice-sar
%pip install -q "fsspec<=2025.3.0" "s3fs<=2025.3.0"
%pip install -q --force-reinstall --no-deps "git+https://github.com/bullocke/nice-sar.git@main"

## Colab Runtime Note

If you already imported `nice_sar` earlier in this Colab runtime, restart the runtime after running the install cell, then run the notebook from the top. Colab can keep the previously imported package in memory even after `%pip install --upgrade`.

In [ ]:
import logging
import os
import tempfile
from pathlib import Path

import matplotlib
if not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from nice_sar.auth.earthdata import (
    get_granule_url,
    get_https_filesystem,
    get_s3_filesystem,
    login,
)
from nice_sar.io.hdf5 import open_nisar, get_frequencies, get_polarizations
from nice_sar.io.products import read_gcov, get_projection_info
from nice_sar.preprocess.calibration import linear_to_db
from nice_sar.viz.rgb import make_rgb, list_rgb_methods
from nice_sar.viz.display import percentile_stretch, to_uint8
from nice_sar.io.geotiff import write_rgb_geotiff_uint8

logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s", force=True)
logger = logging.getLogger(__name__)

# ── Study Areas ──────────────────────────────────────────────────────────────
STUDY_AREAS = {
    "salt_lake_city": (-112.1, 40.5, -111.7, 40.9),
    "cascades_wa": (-122.5, 46.0, -121.5, 47.0),
    "amazon": (-55.0, -3.5, -54.0, -2.5),
}
STUDY_AREA = "salt_lake_city"  # ← change to try other regions
bbox = STUDY_AREAS[STUDY_AREA]
logger.info("Study area: %s  bbox=%s", STUDY_AREA, bbox)

## 2. Authenticate and Load Data

In [ ]:
try:
    login()
except Exception as e:
    logger.error(
        "Authentication failed: %s\n"
        "Ensure credentials are configured via one of:\n"
        "  1. Environment variables: EARTHDATA_USERNAME / EARTHDATA_PASSWORD\n"
        "  2. ~/.netrc with: machine urs.earthdata.nasa.gov login <user> password <pass>\n"
        "  3. Interactive prompt (Colab / Jupyter only)",
        e,
    )
    raise

# S3 on us-west-2, HTTPS elsewhere
if os.environ.get("AWS_DEFAULT_REGION") == "us-west-2":
    fs = get_s3_filesystem()
    granule_access = "s3"
else:
    fs = get_https_filesystem()
    granule_access = "https"

# Search for a GCOV granule
import earthaccess

results = earthaccess.search_data(
    short_name="NISAR_L2_GCOV_BETA_V1",
    bounding_box=bbox,
    count=5,
)
granule_url = get_granule_url(results[0], access=granule_access)
logger.info("Using %s granule URL: %s", granule_access.upper(), granule_url)

In [ ]:
# Read HH and HV bands
h5 = open_nisar(granule_url, filesystem=fs)
da_hh = read_gcov(h5, frequency="A", polarization="HH")
da_hv = read_gcov(h5, frequency="A", polarization="HV")
crs, transform, x_coords, y_coords = get_projection_info(h5, frequency="A")

# Extract numpy arrays — use a center subset for speed
hh_linear = da_hh.values
hv_linear = da_hv.values

cy, cx = hh_linear.shape[0] // 2, hh_linear.shape[1] // 2
sz = 512
slc = (slice(cy - sz, cy + sz), slice(cx - sz, cx + sz))
hh_sub = hh_linear[slc]
hv_sub = hv_linear[slc]

# Also prepare dB versions (some methods need dB input)
hh_db = linear_to_db(hh_sub)
hv_db = linear_to_db(hv_sub)

logger.info("Subset shape: %s", hh_sub.shape)
h5.close()

## 3. Available RGB Methods

`list_rgb_methods()` returns the names of all 12 built-in compositing methods.
Methods prefixed with `"standard"` or `"pseudo_natural"` expect **dB** inputs;
all others expect **linear power** inputs.

In [ ]:
methods = list_rgb_methods()
logger.info("Available RGB methods (%d):", len(methods))
for m in methods:
    logger.info("  • %s", m)

## 4. Generate a Single RGB Composite

Use `make_rgb()` to generate a composite. It dispatches to the correct function
based on the `method` name. Returns `(rgb_array, band_descriptions)`.

In [ ]:
# "vegetation_green" expects linear power inputs
rgb, band_desc = make_rgb(hh_sub, hv_sub, method="vegetation_green")

logger.info("RGB shape: %s, range: [%.3f, %.3f]", rgb.shape, np.nanmin(rgb), np.nanmax(rgb))
logger.info("Band descriptions: %s", band_desc)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(rgb)
ax.set_title("Vegetation Green RGB")
ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Gallery — Compare All 12 Methods

Loop through all methods, passing the correct input type (dB vs linear).

In [ ]:
# Methods that expect dB inputs (check the function signatures)
DB_METHODS = {"standard", "pseudo_natural"}

ncols = 4
nrows = (len(methods) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes_flat = axes.ravel()

for idx, method_name in enumerate(methods):
    if method_name in DB_METHODS:
        rgb, desc = make_rgb(hh_db, hv_db, method=method_name)
    else:
        rgb, desc = make_rgb(hh_sub, hv_sub, method=method_name)

    axes_flat[idx].imshow(rgb)
    axes_flat[idx].set_title(method_name, fontsize=11)
    axes_flat[idx].axis("off")

# Hide unused axes
for idx in range(len(methods), len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle("NISAR GCOV — 12 RGB Composite Methods", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Export RGB Composite to GeoTIFF

Convert an RGB result to uint8 and write a 3-band GeoTIFF with
`write_rgb_geotiff_uint8()`.

In [ ]:
# Generate a full-resolution composite for export
rgb_full, desc_full = make_rgb(hh_linear, hv_linear, method="vegetation_green")
rgb_uint8 = to_uint8(rgb_full)

logger.info("RGB uint8 shape: %s, dtype: %s", rgb_uint8.shape, rgb_uint8.dtype)

# Stack to (3, H, W) for the GeoTIFF writer
rgb_stack = np.moveaxis(rgb_uint8, -1, 0)  # (H, W, 3) → (3, H, W)

with tempfile.TemporaryDirectory() as tmpdir:
    out_path = Path(tmpdir) / "nisar_gcov_vegetation_green.tif"
    write_rgb_geotiff_uint8(
        rgb_stack,
        out_path,
        transform=transform,
        crs=crs,
        band_descriptions=desc_full,
    )
    logger.info("Wrote: %s (%.1f MB)", out_path.name, out_path.stat().st_size / 1e6)

## 7. Customize Gamma Correction

Each method accepts a `gamma` parameter for power-law contrast adjustment.
Lower gamma = brighter midtones, higher gamma = more contrast.

In [ ]:
gammas = [0.3, 0.5, 0.7, 1.0]
fig, axes = plt.subplots(1, len(gammas), figsize=(5 * len(gammas), 5))

for ax, g in zip(axes, gammas):
    rgb_g, _ = make_rgb(hh_sub, hv_sub, method="vegetation_green", gamma=g)
    ax.imshow(rgb_g)
    ax.set_title(f"gamma={g}")
    ax.axis("off")

fig.suptitle("Effect of Gamma on Vegetation Green Composite", fontsize=13)
plt.tight_layout()
plt.show()